# Laptop Price Prediction
**SAiR Regression Capstone**

## 01. Project Overview

**Overview.** This project predicts the price of a laptop (`Price_euros`) from its listed
specifications: manufacturer, type, screen, CPU, GPU, memory, storage, weight and operating
system. It is a supervised regression problem on the Kaggle *Laptop Price Prediction*
dataset (1,303 listings, 13 columns).

**Objective.** Build a price estimator and measure its error honestly, including for laptop
models it has never seen.

**Key activities.**
- Assess data quality and remove redundant listings.
- Analyse the target and the features on the training set only.
- Extract features inside a scikit-learn pipeline, so preprocessing is fitted on training
  folds only.
- Compare feature sets and models with repeated cross-validation, tune the strongest
  candidates, and evaluate once on a held-out test set.
- Save the final model, serve it in a Streamlit app, and track the experiments with MLflow.

**Evaluation.** MAE in euros is the primary metric; RMSE, R² and median absolute error are
reported alongside it. Two validation protocols are used: random K-fold, and K-fold grouped
by product (all rows of a product stay in the same fold).

**Scope.** The model learns from a fixed set of historical listings and is a demonstration,
not a pricing service. Estimates above roughly €2,500 are less reliable because very few
laptops in that range are available for training .

## 02. Environment & Configuration

**Overview.** Imports, display settings, the project configuration, and the library versions
used for the results in this notebook.

**Objective.** Keep every setting in one place and make the results reproducible.

**Key activities.**
- Import libraries once, here; later sections add nothing to this cell.
- Load `ProjectConfig` from `utils.py`, the single source of truth for the split, the
  cross-validation and the encoding settings.
- Record the library versions.


In [ ]:
import json
import textwrap
import warnings
from dataclasses import replace
from importlib.metadata import version

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.stats import loguniform, randint
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, cross_val_predict

from utils import (
    EXPERIMENTS_DIR, FEATURE_SETS, ID_COLUMN, MODEL_FILENAME, MODELS_DIR, OPTIONS_FILENAME,
    RAW_DATA_DIR, RAW_FEATURE_COLUMNS, TARGET, ExperimentTracker, LaptopFeatureExtractor,
    ProjectConfig, build_input_options, build_laptop_row, build_model_metadata, build_pipeline,
    candidate_models, cross_validate_model, cv_metrics, error_by_price_range, load_data,
    log_target, make_cv, regression_metrics, remove_duplicate_listings, save_artifacts,
    search_space, split_data, summarize_cv, tune_model,
)


# Known third-party noise (seaborn 0.13 with matplotlib 3.11); nothing from this project is hidden.
warnings.filterwarnings("ignore", message="vert: bool was deprecated")
warnings.filterwarnings("ignore", message="The set_bad function will be deprecated")
warnings.filterwarnings("ignore", message="The ``noload`` loader strategy is deprecated")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [ ]:
config = ProjectConfig()
print(config)

for package in ["scikit-learn", "pandas", "numpy", "scipy", "mlflow", "streamlit", "joblib"]:
    print(f"{package:14s} {version(package)}")

## 03. Data Loading & Initial Inspection

**Overview.** Load the raw CSV and look at its shape, columns, types and a few rows.

**Objective.** Understand what each column contains before judging its quality.

**Key activities.**
- Load the single CSV found in `data/raw/` (stop if there are none or several).
- Inspect shape, dtypes, summary statistics, and the first and last rows.

**Initial observations.**
- 1,303 rows and 13 columns; `Price_euros` is the target and `laptop_ID` is an identifier.
- Only `Inches` and `Price_euros` are numeric. RAM, weight, storage and screen resolution
  arrive as text with units (`8GB`, `1.37kg`, `256GB SSD + 1TB HDD`, `IPS Panel Full HD
  1920x1080`) and must be parsed.
- `Product` is almost unique per row (618 values), and `Cpu` has 118 values, so raw strings
  are poor categorical features.

In [ ]:
csv_files = list(RAW_DATA_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {RAW_DATA_DIR}")
if len(csv_files) > 1:
    raise ValueError(
        f"Multiple CSV files found: {[f.name for f in csv_files]}. "
        "Please select the dataset explicitly."
    )

DATA_FILE = csv_files[0]
raw_df = load_data(DATA_FILE.name)
print(f"Dataset shape: {raw_df.shape}")

In [ ]:
display(raw_df.head())
display(raw_df.tail())

In [ ]:
raw_df.info()

In [ ]:
display(raw_df.describe(include="all").T)

## 04. Data Quality Assessment

**Overview.** Check completeness, duplicates, text consistency, encoding and unit formats, then
remove redundant listings.

**Objective.** Find problems that would silently corrupt the features or leak across the
train/test split, before any modelling.

**Key activities.**
- Missing values, duplicates, and the structure of `laptop_ID`.
- Case, whitespace and spelling variants in text columns.
- File encoding, and the formats of `Ram`, `Weight`, `Memory`, `ScreenResolution`, `OpSys`.
- Remove duplicate listings and re-check the remaining value distributions.

**Findings and decisions.**
- No missing values and no exact duplicates. Ignoring `laptop_ID`, 28 rows are redundant:
  14 listings appear three times with identical specifications and price, always 14 IDs apart
  (IDs 1279–1320). They are removed before the split, so copies cannot land in both train and
  test. This leaves 1,275 rows.
- `laptop_ID` is unique and sequential and carries no price signal (Spearman 0.07 with price).
  It is excluded from the features.
- Case variants exist in `Product` (42 groups), `Cpu` (2) and `Gpu` (4), and 14 `Gpu` values
  have trailing whitespace. Text normalisation is done inside the feature extractor, so
  training and inference share the same logic.
- The CSV is not UTF-8; `latin-1` decodes accented names correctly (`Portégé`).
- `Ram` and `Weight` have consistent units and parse without errors. `Memory` mixes four
  storage types and up to two drives per laptop. `ScreenResolution` packs panel type, touchscreen
  flag and pixel size into one string.
- `OpSys` has both `macOS` (13 rows) and `Mac OS X` (8 rows). They are merged into one category;
  this is a domain-based decision and was not tested by cross-validation.

These checks use the whole file because they describe the data, not learned statistics. The
target is touched only to confirm that `laptop_ID` carries no price signal.

In [ ]:
feature_cols = [c for c in raw_df.columns if c != ID_COLUMN]

print("Missing values:\n", raw_df.isna().sum()[raw_df.isna().sum() > 0])
print("\nExact duplicates:", raw_df.duplicated().sum())
print("Duplicates excluding laptop_ID:", raw_df.duplicated(subset=feature_cols).sum())

print("\nlaptop_ID unique:", raw_df[ID_COLUMN].is_unique)
print("laptop_ID monotonic increasing:", raw_df[ID_COLUMN].is_monotonic_increasing)
print("laptop_ID vs Price (Spearman):", round(raw_df[ID_COLUMN].corr(raw_df[TARGET], method="spearman"), 3))

for col in raw_df.select_dtypes(include="str").columns:
    stripped = raw_df[col].str.strip()
    print(f"{col}: whitespace issues = {(raw_df[col] != stripped).sum()}, "
          f"case-insensitive unique = {raw_df[col].str.lower().str.strip().nunique()} vs {raw_df[col].nunique()}")

print("\nNon-ASCII chars in Product:", raw_df["Product"].str.contains(r"[^\x00-\x7F]").sum())

In [ ]:
for col in ["Ram", "Weight", "Memory", "ScreenResolution", "OpSys"]:
    print(f"\n{col}:", sorted(raw_df[col].unique())[:40])

In [ ]:
dup_mask = raw_df.duplicated(subset=feature_cols, keep=False)
dup_groups = raw_df[dup_mask].groupby(feature_cols)[ID_COLUMN].agg(list)

print("Group sizes:\n", dup_groups.map(len).value_counts().to_string())
print("\nID gaps within groups:\n", dup_groups.map(lambda ids: tuple(np.diff(ids))).value_counts().to_string())
print("\nID range of duplicated rows:", raw_df.loc[dup_mask, ID_COLUMN].min(), "-", raw_df.loc[dup_mask, ID_COLUMN].max())

In [ ]:
for col in ["Product", "Cpu", "Gpu"]:
    normalized = raw_df[col].str.strip().str.lower()
    variants = raw_df.groupby(normalized)[col].unique()
    variants = variants[variants.map(len) > 1]
    print(f"\n{col}: {len(variants)} groups with >1 spelling")
    print(variants.head(6).map(lambda vals: [repr(v) for v in vals]).to_string())

In [ ]:
print(raw_df.loc[raw_df["Product"].str.contains(r"[^\x00-\x7F]"), "Product"].map(repr).to_string())
for encoding in ("utf-8", "latin-1"):
    try:
        pd.read_csv(DATA_FILE, encoding=encoding)
        print(encoding, "-> OK")
    except UnicodeDecodeError as error:
        print(encoding, "-> FAILED:", error.reason)

In [ ]:
print("Weight all end with 'kg':", raw_df["Weight"].str.endswith("kg").all())
weight = pd.to_numeric(raw_df["Weight"].str.removesuffix("kg"), errors="coerce")
print(weight.describe().round(2).to_string(), "| unparsed:", weight.isna().sum())
print(repr(raw_df.loc[raw_df["ScreenResolution"].str.contains("Touch"), "ScreenResolution"].iloc[0]))
print("\nOpSys counts:\n", raw_df["OpSys"].value_counts().to_string())

In [ ]:
clean_df = remove_duplicate_listings(raw_df)
assert not clean_df.duplicated(subset=feature_cols).any()
print(f"Removed {len(raw_df) - len(clean_df)} rows -> {clean_df.shape}")

In [ ]:
print(clean_df["Inches"].describe().round(2).to_string())
print("\nInches counts:\n", clean_df["Inches"].value_counts().sort_index().to_string())
print("\nRam counts:\n", clean_df["Ram"].value_counts().to_string())

memory_counts = clean_df["Memory"].value_counts()
print("\nRare Memory values (<=3 rows):\n", memory_counts[memory_counts <= 3].to_string())

product = clean_df["Product"].str.strip().str.lower()
product_sizes = product.map(product.value_counts())
print("\nUnique products (normalized):", product.nunique(), "of", len(clean_df), "rows")
print("Rows whose product appears in more than one row:", (product_sizes > 1).sum())

## 05. Target Analysis

**Overview.** Distribution of `Price_euros`, its extreme values, and whether a log transform
is worth testing. This analysis is descriptive: it runs on the cleaned dataset before the
split and fits nothing.

**Objective.** Choose how to treat expensive laptops and how to split the data.

**Key activities.**
- Summary statistics, skewness and kurtosis on the raw and `log1p` scale.
- IQR upper fence and inspection of the listings above it.

**Findings and decisions.**
- n = 1,275; mean €1,135, median €989, range €174–€6,099.
- Right-skewed with a heavy tail: skewness 1.51 and excess kurtosis 4.34, falling to -0.18
  and -0.46 after `log1p`.
- 28 listings (2.2%) exceed the IQR fence (€2,828): 17 gaming, 6 workstation, 3 notebook,
  2 ultrabook. They are high-end hardware (GTX 1080, Xeon/Quadro, 32–64 GB RAM), not data
  errors, so they are kept. Only four listings cost more than €4,000.
- Whether to train on `log1p(price)` is decided by cross-validated MAE in euros (sections
  10–11), not by the shape of the distribution.
- The split is stratified on price deciles so the expensive tail appears in both sets.

In [ ]:
price = clean_df[TARGET]
print(price.describe().round(2).to_string())

In [ ]:
log_price = np.log1p(price)

summary = pd.DataFrame({
    "raw": [price.skew(), price.kurt()],
    "log1p": [log_price.skew(), log_price.kurt()],
}, index=["skewness", "excess kurtosis"]).round(3)
print(summary.to_string())
print("\nQuantiles:\n", price.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(0).to_string())

q1, q3 = price.quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
outliers = clean_df[price > upper]
print(f"\nIQR upper fence: {upper:.0f} | rows above: {len(outliers)} ({len(outliers) / len(clean_df):.1%})")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(price, kde=True, ax=axes[0, 0]); axes[0, 0].set_title("Price (EUR)")
sns.histplot(log_price, kde=True, ax=axes[0, 1]); axes[0, 1].set_title("log1p(Price)")
sns.boxplot(x=price, ax=axes[1, 0]); axes[1, 0].set_title("Price boxplot")
stats.probplot(log_price, dist="norm", plot=axes[1, 1]); axes[1, 1].set_title("log1p(Price) Q-Q plot")
plt.tight_layout()
plt.show()

In [ ]:
columns = ["Company", "TypeName", "Cpu", "Gpu", "Ram", "Memory", "Inches", TARGET]
print(outliers.sort_values(TARGET, ascending=False)[columns].head(12).to_string())
print("\nTypeName among outliers:\n", outliers["TypeName"].value_counts().to_string())

## 06. Train/Test Split

**Overview.** An 80/20 split of the cleaned data, stratified on price deciles
(`random_state=42`).

**Objective.** Reserve a test set that plays no part in any modelling decision.

**Key activities.**
- Split with `split_data` from `utils.py`.
- Compare the price distribution of both sets.

**Findings and decisions.**
- 1,020 training and 255 test laptops with nearly identical price statistics (median €988.5
  vs €989.0).
- All four laptops above €4,000 fall in the training set (test maximum: €3,890), so the test
  set cannot measure performance at the very top of the range. The split was not changed
  after observing this.
- 859 of the 1,275 rows (67%) share a product name with another row (section 04). A random
  split therefore lets train and test share product families, and the test score reflects
  familiar products. Grouped cross-validation (sections 10–13) measures unseen products.
- From here on, exploration and every fitted step use the training set only; the test set
  is used for the final evaluation and interpretation .

In [ ]:
train_df, test_df = split_data(clean_df, config)

print(f"train: {train_df.shape} | test: {test_df.shape}")
comparison = pd.DataFrame({
    "train": train_df[TARGET].describe(),
    "test": test_df[TARGET].describe(),
}).round(1)
print(comparison.to_string())
print(f"rows > {upper:.0f} EUR -> train: {(train_df[TARGET] > upper).sum()} "
      f"| test: {(test_df[TARGET] > upper).sum()}")

## 07. Exploratory Data Analysis 

**Overview.** Univariate, bivariate and multivariate exploration of the 1,020 training laptops,
using the same feature-extraction code as the model.

**Objective.** Decide which representations of CPU, GPU, storage, OS and product are worth
testing, and find redundancy and confounding before modelling.

**Key activities.** Distributions of the specifications, the association of each feature with
price, and the relationships between features.

**Decisions carried forward.** `laptop_ID`, raw `Product` and `cpu_brand` are not used as
features. CPU detail (suffix, generation, clock), GPU model and product line are candidates;
they are adopted only if cross-validation supports them .

In [ ]:

features = LaptopFeatureExtractor(merge_mac=False).fit_transform(train_df)
eda = features.join(train_df[["Product", "Cpu", "Gpu", TARGET]])

# Exploration-only columns, not used by the model
eda["cpu_gen"] = pd.to_numeric(eda["cpu_gen"], errors="coerce")   # "none" -> NaN (Apple CPUs)
eda["cpu_brand"] = eda["Cpu"].str.strip().str.split().str[0]
eda["gpu_brand"] = eda["Gpu"].str.strip().str.split().str[0]
eda["cpu_no_model"] = eda["Cpu"].str.strip().str.fullmatch(r"Intel Core (i[357]|M) [\d.]+GHz")

num_cols = ["inches", "ram_gb", "weight_kg", "ssd_gb", "hdd_gb", "hybrid_gb",
            "flash_gb", "touchscreen", "ips", "res_w", "res_h"]

### 07.1 Univariate analysis

**Investigated.** Numeric specifications, storage types, raw CPU/GPU strings and product names.

**Findings.**
- Typical laptop: 15.6", 8 GB RAM (maximum 64 GB), Full HD (median width 1,920).
- SSD appears in 666 of 1,020 laptops, HDD in 446, flash in 60 and hybrid in only 9. Every
  `Memory` value parsed.
- `Cpu` (106 values) and `Gpu` (99 raw strings) spell the same hardware differently
  (`GTX1080` vs `GTX 1080`; `A10-Series 9620P` vs `A10-Series A10-9620P`). Two GPU strings
  contain the literal text `<U+039C>`, an export artifact. Normalisation reduces 99 GPU
  strings to 91.
- The first word of `Product` gives 185 product lines; the 23 lines with at least 10 laptops
  cover 73.5% of the rows.

**Implication.** Raw CPU and GPU strings cannot be used as categories, so family- and
normalised-model-level features are extracted. Hybrid and flash storage are too rare to carry
much signal alone.

In [ ]:
print(eda[num_cols].describe().round(2).T.to_string())

In [ ]:
storage_cols = ["ssd_gb", "hdd_gb", "hybrid_gb", "flash_gb"]
print("Non-zero rows:", {c: int((eda[c] > 0).sum()) for c in storage_cols})
print("Rows with no storage parsed:", int((eda[storage_cols].sum(axis=1) == 0).sum()))

In [ ]:
for col in ["Cpu", "Gpu"]:
    values = sorted(eda[col].str.strip().unique())
    print(f"\n{col} ({len(values)} unique in train):")
    print(textwrap.fill(" | ".join(values), width=160))

In [ ]:
counts = eda["product_line"].value_counts()
print("unique first tokens:", counts.size,
      "| tokens with >= 10 rows:", int((counts >= 10).sum()),
      "| share of rows covered:", round(counts[counts >= 10].sum() / len(eda), 3))
print("rows whose first token contains a digit:", int(counts[counts.index.str.contains(r"\d")].sum()))

### 07.2 Bivariate analysis

**Investigated.** The association of each feature with price: Spearman correlation for numeric
features, median price for categories.

**Findings.**
- Strongest numeric associations: RAM (0.77), SSD capacity (0.70), screen width (0.64) and
  height (0.62); weaker: IPS (0.29) and touchscreen (0.21). Screen size (-0.08) and weight
  (-0.05) show almost none. Price spread grows with RAM.
- `type_name` separates price levels clearly (median €455 for netbooks, €1,993 for
  workstations). Four companies cover 78% of the laptops; 11 companies have 8 or fewer.
- CPU family orders prices (median €1,427 for Core i7, €955 for i5, €499 for i3, €299 for
  Celeron). Clock speed correlates with price at 0.52 overall but only between -0.03 and 0.40
  inside a family, so the family carries most of its signal. The suffix separates Core i prices
  (U €940, HQ €1,474, HK €2,799). Generation is positively related to price within Core i3
  (0.33) and negatively within i5 (-0.15) and i7 (-0.24).
- GPU family orders prices (Quadro €2,182, GTX €1,370, Radeon €659). Within GTX, price rises
  with the model, but model numbers are not an ordinal scale (the 970M and 980M cost more than
  the 1060).
- Product line matters within a company (ThinkPad €1,650 vs IdeaPad €574; Inspiron €819 vs
  XPS €1,639 vs Alienware €2,841).
- One Asus G701VO (32 GB RAM, €1,279) costs far less than every other 32 GB laptop
  (at least €2,639). It is kept and re-examined with out-of-fold residuals (section 15).

**Implication.** RAM, storage and resolution are the main numeric drivers. CPU detail, GPU model
and product line are plausible additions whose value must be measured, not assumed.

In [ ]:
print("Spearman with price:\n",
      eda[num_cols + [TARGET]].corr(method="spearman")[TARGET]
      .drop(TARGET).sort_values(ascending=False).round(3).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.boxplot(data=eda, x="ram_gb", y=TARGET, ax=axes[0, 0]); axes[0, 0].set_title("Price by RAM (GB)")
sns.scatterplot(data=eda, x="weight_kg", y=TARGET, alpha=0.4, ax=axes[0, 1]); axes[0, 1].set_title("Weight vs Price")
sns.scatterplot(data=eda, x="ssd_gb", y=TARGET, alpha=0.4, ax=axes[1, 0]); axes[1, 0].set_title("SSD capacity vs Price")
sns.boxplot(data=eda, x="type_name", y=TARGET, ax=axes[1, 1])
axes[1, 1].tick_params(axis="x", rotation=45); axes[1, 1].set_title("Price by type")
plt.tight_layout(); plt.show()

In [ ]:
for col in ["company", "type_name", "opsys", "cpu_brand", "gpu_brand"]:
    summary_df = (eda.groupby(col)[TARGET].agg(["count", "median", "mean"]).round(0)
                  .sort_values("count", ascending=False))
    print(f"\n{col}:\n", summary_df.to_string())

In [ ]:
columns = ["company", "Product", "type_name", "Cpu", "Gpu", "ram_gb", "ssd_gb", TARGET]
print(eda[eda["ram_gb"] >= 32].sort_values(TARGET)[columns].head(8).to_string())

In [ ]:
print(eda.groupby("cpu_family")[TARGET].agg(["count", "median"])
      .sort_values("count", ascending=False).round(0).to_string())

print("\nSpearman(clock, price) overall:", round(eda["cpu_ghz"].corr(eda[TARGET], method="spearman"), 3))
for family, group in eda.groupby("cpu_family"):
    if len(group) >= 30:
        print(f"  within {family} (n={len(group)}):", round(group["cpu_ghz"].corr(group[TARGET], method="spearman"), 3))

core_i = eda[eda["cpu_family"].str.startswith("Core i")]
print("\nCore i suffix:\n", core_i.groupby("cpu_suffix")[TARGET].agg(["count", "median"])
      .sort_values("count", ascending=False).round(0).to_string())

In [ ]:
core_i = eda[eda["cpu_family"].str.startswith("Core i")]
print("Core i rows without generation:", int(core_i["cpu_gen"].isna().sum()),
      "| Apple among Core i:", int((core_i["company"] == "Apple").sum()))

for family in ["Core i3", "Core i5", "Core i7"]:
    g = core_i[(core_i["cpu_family"] == family) & core_i["cpu_gen"].notna()]
    print(f"\n{family}: Spearman(gen, price) = {g['cpu_gen'].corr(g[TARGET], method='spearman'):.3f} | "
          f"Spearman(clock, gen) = {g['cpu_ghz'].corr(g['cpu_gen'], method='spearman'):.3f}")
    print(g.groupby("cpu_gen")[TARGET].agg(["count", "median"]).round(0).to_string())

In [ ]:
print("Gpu containing '<U+':", eda["Gpu"].str.contains("<U+", regex=False).sum())
print("unique raw:", eda["Gpu"].nunique(), "-> normalized:", eda["gpu_model"].nunique())
print(eda.groupby("gpu_family")[TARGET].agg(["count", "median"])
      .sort_values("count", ascending=False).round(0).to_string())
print("\nValues in 'other' families:\n",
      eda.loc[eda["gpu_family"].str.contains("other|Other"), "gpu_model"].value_counts().to_string())

gtx = eda[eda["gpu_family"] == "Nvidia GTX"]
print("\nGTX models with >= 5 rows:\n",
      gtx.groupby("gpu_model")[TARGET].agg(["count", "median", "min", "max"])
      .query("count >= 5").sort_values("median").round(0).to_string())

In [ ]:
print(eda.groupby("product_line")[TARGET].agg(["count", "median"])
      .query("count >= 10").sort_values("count", ascending=False).round(0).to_string())

### 07.3 Multivariate analysis

**Investigated.** Redundancy and confounding between features.

**Findings.**
- RAM, SSD capacity and resolution are mutually correlated (0.55–0.61), so they partly encode
  one "device tier" signal. Screen size and weight correlate at 0.88, and screen width and
  height are almost perfectly rank-correlated.
- The 18 Apple laptops are exactly the macOS / Mac OS X rows and exactly the CPUs listed
  without a model number, so for Apple `opsys` and the missing CPU details add nothing beyond
  `company`.
- Windows 7 (40 laptops) looks expensive (median €1,482 vs €1,029 for Windows 10), but 37 of
  them are business lines (EliteBook, ThinkPad, ZBook, Latitude, ...) and it includes 7 of the 23
  workstations: much of the effect is product mix.
- Holding the CPU suffix fixed does not remove the negative generation effect (U-suffix Core i5
  -0.16, i7 -0.21). Generation and suffix are also entangled: generation 8 appears only with
  the U suffix. The cause was not investigated.
- The cheap 32 GB Asus belongs to the ROG G701 family, whose other models cost €2,799–€3,975.

**Implication.** OS and CPU-generation effects are confounded with company and product line, so
they are tested by ablation rather than interpreted directly.

In [ ]:
corr = eda[num_cols].corr(method="spearman")
plt.figure(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7})
plt.title("Spearman correlation between numeric features")
plt.tight_layout()
plt.show()

In [ ]:
print(pd.crosstab(eda["company"] == "Apple", eda["opsys"].isin(["macOS", "Mac OS X"]),
                  rownames=["is_apple"], colnames=["mac_os"]).to_string())
print("\nNo-model CPUs vs Apple:\n", pd.crosstab(eda["cpu_no_model"], eda["company"] == "Apple").to_string())

w7 = eda[eda["opsys"] == "Windows 7"]
print("\nWindows 7 by type:\n", w7["type_name"].value_counts().to_string())
print("\nWindows 7 by company:\n", w7["company"].value_counts().to_string())
print("\nWindows 7 by first word of Product:\n", w7["Product"].str.split().str[0].value_counts().head(8).to_string())

print("\nSame product family as the cheap 32GB laptop:")
print(eda.loc[eda["Product"].str.contains("G701"), ["Product", "Gpu", "ram_gb", TARGET]].to_string())

In [ ]:
core_i = eda[eda["cpu_family"].str.startswith("Core i")]
print(pd.crosstab(core_i["cpu_gen"], core_i["cpu_suffix"]).to_string())

u_only = core_i[(core_i["cpu_suffix"] == "U") & core_i["cpu_gen"].notna()]
for family in ["Core i5", "Core i7"]:
    g = u_only[u_only["cpu_family"] == family]
    print(f"\n{family} (U only, n={len(g)}): Spearman(gen, price) = "
          f"{g['cpu_gen'].corr(g[TARGET], method='spearman'):.3f}")
    print(g.groupby("cpu_gen")[TARGET].agg(["count", "median"]).round(0).to_string())

## 08. Feature Engineering & Preprocessing

**Overview.** The raw text columns are converted into numeric and categorical features by
`LaptopFeatureExtractor`, a stateless scikit-learn transformer defined in `utils.py`. Learned
preprocessing (scaling, one-hot encoding) is left to the pipeline (section 09).

**Objective.** One feature definition shared by the notebook, the cross-validation and the
Streamlit app, so training and inference cannot diverge.

**Key activities.**

| Raw column | Derived features |
|---|---|
| `Inches`, `Ram`, `Weight` | `inches`, `ram_gb`, `weight_kg` |
| `Memory` | `ssd_gb`, `hdd_gb`, `hybrid_gb`, `flash_gb` |
| `ScreenResolution` | `touchscreen`, `ips`, `res_w`, `res_h`, `ppi` |
| `Cpu` | `cpu_family`, `cpu_suffix`, `cpu_gen`, `cpu_ghz` |
| `Gpu` | `gpu_family`, `gpu_model` |
| `Company`, `TypeName`, `OpSys` | `company`, `type_name`, `opsys` |
| `Product` | `product_line` (lower-cased first word) |

**Decisions.**
- Parsing is strict: a value that cannot be parsed raises an error instead of becoming NaN.
  The training features contain no missing values.
- Storage: 1 TB = 1,024 GB, and drives of the same type are summed (for example two SSDs).
- `Mac OS X` is merged into `macOS` (a domain decision; it was not cross-validated).
- Whitespace is collapsed, and GPU names are normalised, including the `<U+039C>` artifact.
- The extractor produces 22 candidate columns, and the base feature set uses 15 of them. CPU
  detail, GPU model and product line are tested one by one in section 11. `ppi` is not used by
  any feature set, and `res_h` only feeds the resolution choices in the app.
- Not used: `laptop_ID`, the raw `Product`, `Cpu` and `Gpu` strings (too many distinct values),
  and the exploration-only `cpu_brand`.

In [ ]:
model_features = LaptopFeatureExtractor().fit_transform(train_df)
display(model_features.head())
print(f"features: {model_features.shape} | NaN: {int(model_features.isna().sum().sum())}")

categorical_candidates = ["company", "type_name", "opsys", "cpu_family", "cpu_suffix",
                          "cpu_gen", "gpu_family", "gpu_model", "product_line"]
print(model_features[categorical_candidates].nunique().to_string())
print("Mac OS rows after merge:", model_features["opsys"].value_counts().get("macOS"))

## 09. Pipeline Construction

**Overview.** Every model is a single scikit-learn `Pipeline`: `LaptopFeatureExtractor` →
`ColumnTransformer` (numeric and categorical branches) → estimator. `build_pipeline` in
`utils.py` assembles it for any estimator and feature set.

**Objective.** Make every learned step part of the estimator, so each cross-validation fold
and the final fit learn only from their own training rows.

**Key activities.**
- Define eight feature sets, each adding candidate columns to the base set.
- Fit a Ridge pipeline on each set as a sanity check and count the model inputs.

**Design and leakage prevention.**
- Numeric branch: `StandardScaler` for linear models, pass-through for tree models.
- Categorical branch: `OneHotEncoder`; categories with fewer than 10 training rows are grouped
  into an "infrequent" bucket, and categories unseen at prediction time fall into the same
  bucket instead of raising an error. The threshold is tuned over 5, 10 and 20 in section 13.
- No imputation (no missing values) and no outlier removal.
- The extractor holds no learned state; the scaler and the encoder are fitted inside each
  training fold only, and the test set is never passed to `fit`.

| Feature set | Adds to the base set | Model inputs |
|---|---|---|
| `base` | none | 47 |
| `cpu_suffix` | `cpu_suffix` | 52 |
| `cpu_suffix_gen` | `cpu_suffix`, `cpu_gen` | 56 |
| `cpu_suffix_gen_ghz` | `cpu_suffix`, `cpu_gen`, `cpu_ghz` | 57 |
| `gpu_model` | `gpu_model` | 70 |
| `product_line` | `product_line` | 71 |
| `gpu_product` | `gpu_model`, `product_line` | 94 |
| `gpu_product_cpu` | `gpu_model`, `product_line`, `cpu_suffix`, `cpu_gen`, `cpu_ghz` | 104 |

The check below fits on the training set only to count inputs and confirm finite predictions;
it evaluates nothing.

In [ ]:
X_train, y_train = train_df.drop(columns=[TARGET]), train_df[TARGET]

for name, (numeric, categorical) in FEATURE_SETS.items():
    pipe = build_pipeline(Ridge(), numeric, categorical, scale=True,
                          min_category_count=config.min_category_count)
    pipe.fit(X_train, y_train)
    n_inputs = len(pipe.named_steps["preprocess"].get_feature_names_out())
    assert np.isfinite(pipe.predict(X_train)).all()
    print(f"{name:22s} -> {n_inputs} model inputs")

## 10. Evaluation Protocol & Baseline

**Overview.** The validation protocols used for every comparison, and the first reference
models: two dummy regressors and a Ridge regression on the base features.

**Objective.** Fix the yardstick before comparing anything, so that every difference between
two models is measured on identical folds.

**Key activities.**
- Random protocol: repeated K-fold on the training set, 5 folds × 3 repeats = 15 folds.
- Grouped protocol: K-fold where all rows of the same product (normalised `Product`) stay in
  the same fold, 5 folds × 3 repeats = 15 folds. The training set has 491 distinct products in
  1,020 rows.
- Metrics: MAE in euros is the primary metric, with RMSE, R², median absolute error and fit
  time. Models trained on `log1p(price)` are scored in euros after inverting the transform.
- Baselines: mean and median dummies, and Ridge with a raw and a `log1p` target.

**Findings and decisions.**
- Dummy regressors give an MAE of €537 (mean) and €525 (median). Ridge on the base features
  reaches €237 ± 20 (R² 0.76).
- A `log1p` target lowers the median error (€135 vs €166) but inflates large errors (RMSE €426
  vs €340). The paired MAE difference is -€4.3 ± €17.3 and the log target wins in 7 of 15 folds,
  so it brings no gain here.
- Decision: models are trained on the raw price. The log target is re-tested with a random
  forest in section 11.
- Random folds measure error on laptops similar to those seen in training; grouped folds
  measure error on unseen products and are the stricter estimate.

In [ ]:
product_groups = X_train["Product"].str.strip().str.lower()
print("distinct products:", product_groups.nunique(), "| rows:", len(product_groups))

In [ ]:
base_numeric, base_categorical = FEATURE_SETS["base"]


def ridge_pipeline():
    return build_pipeline(Ridge(), base_numeric, base_categorical, scale=True,
                          min_category_count=config.min_category_count)


models = {
    "Dummy (mean)": DummyRegressor(strategy="mean"),
    "Dummy (median)": DummyRegressor(strategy="median"),
    "Ridge, raw target": ridge_pipeline(),
    "Ridge, log1p target": log_target(ridge_pipeline()),
}
results = {name: cross_validate_model(m, X_train, y_train, config) for name, m in models.items()}

summary = summarize_cv(results)
print(summary[["mae_mean", "mae_std", "rmse_mean", "rmse_std", "r2_mean", "medae_mean",
               "fit_seconds_mean"]].round(2).to_string())

diff = results["Ridge, raw target"]["mae"] - results["Ridge, log1p target"]["mae"]
print(f"\nPaired MAE difference (raw - log), positive = log better: "
      f"mean {diff.mean():.1f} EUR, std {diff.std():.1f}, log wins {(diff > 0).sum()}/{len(diff)} folds")

## 11. Feature-Set Experiments

**Overview.** Ablation of the candidate features on top of the base set (section 09), plus a
check of the target transform. A linear model (Ridge) and a tree model (RandomForest) are used
under both validation protocols, so conclusions do not depend on one model family or on shared
product families.

**Objective.** Freeze the feature set and the target before comparing models.

**Key activities.**
- Compare all eight feature sets with random cross-validation, then the five main ones with
  grouped cross-validation.
- Re-test the `log1p` target with RandomForest (Ridge was tested in section 10).
- Test whether CPU detail adds value on top of GPU model and product line.

**Findings and decisions.**
- GPU model and product line each help, and both together help most. Random CV, MAE gain over
  the base set: Ridge -€14.1 (GPU model), -€33.7 (product line), -€42.1 (both); RandomForest
  -€8.2, -€9.9 and -€17.0.
- The gains persist when whole products are held out (Ridge -€14.3, -€34.0, -€41.4; RandomForest
  -€9.6, -€14.9, -€20.8), in 15 of 15 folds each. They are therefore not memorisation of product
  siblings, and both features are adopted.
- Holding out whole products raises the base MAE by 4.9% for Ridge and 14.1% for RandomForest, so
  random K-fold scores are optimistic for unseen products. Both protocols are reported from here on.
- CPU detail (suffix, generation, clock) gives small gains on top of GPU model and product line:
  1.5–2.4% of MAE. The pre-defined criterion was that at least 12 of 15 folds improve and the mean
  gain exceeds its standard deviation. Ridge meets it (13/15 in both protocols); RandomForest does
  not (11/15 and 10/15, gain below its standard deviation). Because the final candidates are
  tree ensembles and CPU detail would add 10 model inputs, CPU family alone is kept.
- Target: `log1p` does not help RandomForest either (-€0.4 ± 4.4, 7/15 folds), so models are
  trained on the raw price.
- Frozen feature set: `gpu_product` (base features plus GPU model and product line), raw price.

In [ ]:
def make_ridge(numeric, categorical):
    return build_pipeline(Ridge(), numeric, categorical, scale=True,
                          min_category_count=config.min_category_count)


def make_forest(numeric, categorical):
    forest = RandomForestRegressor(n_estimators=300, random_state=config.random_state, n_jobs=-1)
    return build_pipeline(forest, numeric, categorical, scale=False,
                          min_category_count=config.min_category_count)


def run_ablation(make_model, names=None, groups=None):
    names = names or list(FEATURE_SETS)
    return {name: cross_validate_model(make_model(*FEATURE_SETS[name]), X_train, y_train, config, groups)
            for name in names}


def ablation_table(results):
    base_mae = results["base"]["mae"]
    rows = {}
    for name, folds in results.items():
        gain = base_mae - folds["mae"]
        rows[name] = {
            "mae_mean": folds["mae"].mean(), "mae_std": folds["mae"].std(),
            "rmse_mean": folds["rmse"].mean(),
            "gain_vs_base": gain.mean(), "gain_std": gain.std(),
            "wins": f"{int((gain > 0.01).sum())}/{len(gain)}",
        }
    return pd.DataFrame(rows).T

In [ ]:
# Random K-fold: all eight feature sets. Results are stored and reused below.
ablation = {}
for label, make in [("Ridge", make_ridge), ("RandomForest", make_forest)]:
    ablation[(label, "random")] = run_ablation(make)
    print(f"{label} | random KFold\n", ablation_table(ablation[(label, "random")]).round(2).to_string(), "\n")

In [ ]:
# Grouped by product: the five main feature sets.
subset = ["base", "gpu_model", "product_line", "gpu_product", "gpu_product_cpu"]
for label, make in [("Ridge", make_ridge), ("RandomForest", make_forest)]:
    ablation[(label, "grouped")] = run_ablation(make, subset, product_groups)
    print(f"{label} | grouped by Product\n", ablation_table(ablation[(label, "grouped")]).round(2).to_string(), "\n")

In [ ]:
forest_raw = ablation[("RandomForest", "random")]["base"]
forest_log = cross_validate_model(log_target(make_forest(*FEATURE_SETS["base"])), X_train, y_train, config)
gain = forest_raw["mae"] - forest_log["mae"]
print(f"RF raw: MAE {forest_raw['mae'].mean():.1f} | RMSE {forest_raw['rmse'].mean():.1f} | MedAE {forest_raw['medae'].mean():.1f}")
print(f"RF log: MAE {forest_log['mae'].mean():.1f} | RMSE {forest_log['rmse'].mean():.1f} | MedAE {forest_log['medae'].mean():.1f}")
print(f"Paired MAE gain of log: mean {gain.mean():.1f} EUR, std {gain.std():.1f}, wins {(gain > 0.01).sum()}/{len(gain)}")

In [ ]:
# CPU detail on top of GPU model + product line.
rows = {}
for (label, protocol), res in ablation.items():
    without, with_cpu = res["gpu_product"]["mae"], res["gpu_product_cpu"]["mae"]
    gain = without - with_cpu
    n_wins = int((gain > 0.01).sum())
    rows[(label, protocol)] = {
        "mae_without_cpu": without.mean(), "mae_with_cpu": with_cpu.mean(),
        "gain": gain.mean(), "gain_std": gain.std(),
        "gain_pct": 100 * gain.mean() / without.mean(),
        "wins": f"{n_wins}/{len(gain)}",
        "meets_criterion": n_wins >= 12 and gain.mean() > gain.std(),
    }
print(pd.DataFrame(rows).T.round(2).to_string())

FROZEN_SET = "gpu_product"
frozen_numeric, frozen_categorical = FEATURE_SETS[FROZEN_SET]

## 12. Model Comparison

**Overview.** Nine untuned models are compared on the frozen feature set with random
cross-validation; the five strongest are then re-evaluated with grouped cross-validation.

**Objective.** Select the models worth tuning.

**Key activities.**
- Linear models (LinearRegression, Ridge, Lasso, ElasticNet), tree models (DecisionTree,
  RandomForest, ExtraTrees) and boosting models (GradientBoosting, HistGradientBoosting), each
  with a sensible starting configuration from `candidate_models`.
- Paired MAE differences against the best model over the same 15 folds.

**Findings and decisions.**
- Random CV: GradientBoosting is best (MAE €167.6), then RandomForest (€174.5),
  HistGradientBoosting (€180.9) and ExtraTrees (€182.3). The linear family follows at
  €194–197, about €26–30 behind GradientBoosting, and a single DecisionTree is weakest (€219.4).
- GradientBoosting beats RandomForest by €6.9 ± 7.4 (13/15 folds), which is not conclusive. The four
  linear models are practically equivalent.
- GradientBoosting has the lowest RMSE and R² 0.87; ExtraTrees and RandomForest have the lowest
  median error (€104 and €106).
- Grouped CV raises MAE by about 13% for the boosting and bagging models, 6% for Ridge and 18% for
  ExtraTrees. The ranking of the three leading models is unchanged, and GradientBoosting vs
  RandomForest remains inconclusive (€7.8 ± 11.2, 11/15 folds).
- Decision: tune GradientBoosting, HistGradientBoosting and RandomForest; keep Ridge as the linear
  reference; drop ExtraTrees, which is the most sensitive to shared product families. All
  configurations here are untuned starting values (HistGradientBoosting runs on defaults).

In [ ]:
models = {
    name: build_pipeline(estimator, frozen_numeric, frozen_categorical, scale=scale,
                         min_category_count=config.min_category_count)
    for name, (estimator, scale) in candidate_models(config.random_state).items()
}
portfolio = {name: cross_validate_model(model, X_train, y_train, config) for name, model in models.items()}

summary = summarize_cv(portfolio).sort_values("mae_mean")
print(summary[["mae_mean", "mae_std", "rmse_mean", "r2_mean", "medae_mean", "fit_seconds_mean"]].round(2).to_string())

best = summary.index[0]
print(f"\nPaired MAE difference vs best model ({best}); positive = {best} better")
for name in summary.index[1:]:
    diff = portfolio[name]["mae"] - portfolio[best]["mae"]
    print(f"{name:22s} mean {diff.mean():7.1f} EUR | std {diff.std():5.1f} | {best} wins {(diff > 0.01).sum()}/{len(diff)}")

In [ ]:
finalists = ["GradientBoosting", "RandomForest", "HistGradientBoosting", "ExtraTrees", "Ridge"]
grouped = {name: cross_validate_model(models[name], X_train, y_train, config, product_groups)
           for name in finalists}

summary_grouped = summarize_cv(grouped).sort_values("mae_mean")
print(summary_grouped[["mae_mean", "mae_std", "rmse_mean", "r2_mean", "medae_mean"]].round(2).to_string())

best_grouped = summary_grouped.index[0]
print(f"\nPaired MAE difference vs best grouped model ({best_grouped}); positive = {best_grouped} better")
for name in summary_grouped.index[1:]:
    diff = grouped[name]["mae"] - grouped[best_grouped]["mae"]
    print(f"{name:22s} mean {diff.mean():7.1f} EUR | std {diff.std():5.1f} | {best_grouped} wins {(diff > 0.01).sum()}/{len(diff)}")

## 13. Hyperparameter Optimization

**Overview.** Randomized search for GradientBoosting, HistGradientBoosting and RandomForest
(40 candidates each, MAE objective, 5-fold GroupKFold by product, one repeat) and a grid search
over six alpha values for Ridge. Each best configuration is then re-evaluated with 5 folds × 3
repeats under both protocols.

**Objective.** Check how much tuning adds, using the stricter grouped protocol as the objective,
because random folds reward configurations that exploit sibling configurations of the same product.

**Key activities.**
- Search and re-evaluation (below), and a paired comparison of the tuned models.
- A refinement round around the best boosting configurations.

**Findings and decisions.**

| Model | Random MAE (untuned → tuned) | Grouped MAE (untuned → tuned) | Grouped gain, paired |
|---|---|---|---|
| GradientBoosting | 167.6 → 157.4 | 190.0 → 179.5 | 10.5 ± 13.1 (10/15) |
| HistGradientBoosting | 180.9 → 161.1 | 204.3 → 182.8 | 21.4 ± 14.0 (14/15) |
| RandomForest | 174.5 → 170.6 | 197.8 → 194.3 | 3.5 ± 6.8 (10/15) |
| Ridge (alpha = 3) | 195.1 → 194.0 | 207.3 → 204.5 | 2.9 ± 2.6 (13/15) |

- Only HistGradientBoosting gains clearly (its defaults overfit). For GradientBoosting the grouped
  gain is not conclusive, and for RandomForest and Ridge it is negligible.
- The selected boosting configurations use very small trees (GradientBoosting `max_depth=2` with
  Huber loss; HistGradientBoosting `max_leaf_nodes=10`) and many iterations, which may indicate a
  nearly additive price structure. This was not tested.
- Tuned GradientBoosting and HistGradientBoosting are practically tied (grouped difference
  €3.3 ± 9.9, 10/15 folds); RandomForest and Ridge are clearly behind.
- Refinement round: a finer grid for GradientBoosting and a second search for HistGradientBoosting
  improved grouped MAE by only €2.5 ± 7.7 (8/15 folds) and €1.4 ± 7.2 (10/15). Neither is adopted;
  the first tuned configurations are kept.

**Caveats.** The first repeat of the re-evaluation uses the same folds as the search, so the tuned
grouped scores are slightly optimistic; the held-out test set (section 15) is the unbiased check.
The three randomized searches used one repeat of grouped folds, whereas the Ridge grid and the
refinement searches used three.

In [ ]:
tuned = {}
for name in ["GradientBoosting", "RandomForest", "HistGradientBoosting"]:
    search, summary = tune_model(models[name], search_space(name), X_train, y_train, config,
                                 groups=product_groups, n_iter=40, n_jobs=-1)
    tuned[name] = (search, summary)
    print(f"{name}: best grouped CV MAE {summary['best_cv_mae']:.1f} | candidates {summary['n_candidates']} "
          f"| {summary['search_seconds'] / 60:.1f} min")
    print("  ", {k.replace("model__", ""): (round(v, 4) if isinstance(v, float) else v)
                 for k, v in summary["best_params"].items()})

In [ ]:
ridge_search = GridSearchCV(
    models["Ridge"], {"model__alpha": [0.1, 1, 3, 10, 30, 100]},
    scoring="neg_mean_absolute_error",
    cv=make_cv(config, X_train, product_groups), n_jobs=-1,
).fit(X_train, y_train)
print("Ridge best:", ridge_search.best_params_, "| grouped CV MAE:", round(-ridge_search.best_score_, 1))
tuned["Ridge"] = (ridge_search, None)

In [ ]:
rows = {}
final_cv = {}
for name, (search, _) in tuned.items():
    best = search.best_estimator_
    for protocol, groups in [("random", None), ("grouped", product_groups)]:
        folds = cross_validate_model(best, X_train, y_train, config, groups)
        final_cv[(name, protocol)] = folds
        untuned = portfolio[name] if protocol == "random" else grouped[name]
        gain = untuned["mae"] - folds["mae"]
        rows[(name, protocol)] = {
            "untuned_mae": untuned["mae"].mean(), "tuned_mae": folds["mae"].mean(),
            "tuned_std": folds["mae"].std(), "tuned_rmse": folds["rmse"].mean(),
            "tuned_medae": folds["medae"].mean(),
            "gain": gain.mean(), "gain_std": gain.std(), "wins": f"{int((gain > 0.01).sum())}/{len(gain)}",
        }
print(pd.DataFrame(rows).T.round(2).to_string())

In [ ]:
for protocol in ["random", "grouped"]:
    ref = final_cv[("GradientBoosting", protocol)]["mae"]
    for name in ["HistGradientBoosting", "RandomForest", "Ridge"]:
        diff = final_cv[(name, protocol)]["mae"] - ref
        print(f"{protocol:8s} {name:22s} MAE - GB: mean {diff.mean():6.1f} | std {diff.std():5.1f} "
              f"| GB better in {(diff > 0.01).sum()}/{len(diff)}")

In [ ]:
# Refinement round (a negative result: documented, not adopted)
def fold_mae(search):
    i = search.best_index_
    keys = [k for k in search.cv_results_ if k.startswith("split") and k.endswith("_test_score")]
    return -np.array([search.cv_results_[f"split{k}_test_score"][i] for k in range(len(keys))])


def compare_with_current(name, search):
    current = final_cv[(name, "grouped")]["mae"].to_numpy()
    refined = fold_mae(search)
    diff = current - refined
    print(f"{name} refined: grouped MAE {refined.mean():.1f} (current {current.mean():.1f}) | "
          f"gain {diff.mean():.1f} ± {diff.std(ddof=1):.1f} | refined better in {(diff > 0.01).sum()}/{len(diff)}")
    print("   ", {k.replace("model__", ""): (v.item() if isinstance(v, np.generic) else v)
                  for k, v in search.best_params_.items()})
    rand = cross_validate_model(search.best_estimator_, X_train, y_train, config)
    print(f"    random-CV MAE of refined model: {rand['mae'].mean():.1f} "
          f"(current {final_cv[(name, 'random')]['mae'].mean():.1f})")


gb_base = clone(models["GradientBoosting"]).set_params(**tuned["GradientBoosting"][0].best_params_)
gb_grid = GridSearchCV(
    gb_base,
    {"model__max_depth": [1, 2, 3], "model__learning_rate": [0.1, 0.15, 0.2],
     "model__n_estimators": [400, 800], "model__alpha": [0.9, 0.95]},
    scoring="neg_mean_absolute_error", cv=make_cv(config, X_train, product_groups), n_jobs=-1,
).fit(X_train, y_train)
compare_with_current("GradientBoosting", gb_grid)

hgb_space = {
    "model__loss": ["squared_error"],
    "model__learning_rate": loguniform(0.03, 0.12),
    "model__max_iter": randint(300, 901),
    "model__max_leaf_nodes": randint(4, 17),
    "model__max_depth": [None, 3, 5],
    "model__min_samples_leaf": randint(2, 16),
    "model__l2_regularization": loguniform(0.5, 30),
    "preprocess__cat__min_frequency": [5, 10],
}
hgb_refined, hgb_refined_summary = tune_model(
    models["HistGradientBoosting"], hgb_space, X_train, y_train, config,
    groups=product_groups, n_iter=30, n_repeats=config.n_repeats, n_jobs=-1,
)
compare_with_current("HistGradientBoosting", hgb_refined)

## 14. Final Model Selection

**Overview.** Comparison of the four tuned candidates under both protocols, and selection of the
final model.

**Objective.** Choose the model using only training data, with a rule fixed before the test set
is touched.

**Key activities.**
- Compare MAE, RMSE, median error and fit time of the tuned models.
- Select the model with the lowest grouped CV MAE: the stricter estimate, because the app will
  receive products the model has not seen.

**Decision.**
- GradientBoosting is selected (grouped MAE €179.5 ± 28.1, random €157.4).
- HistGradientBoosting is practically tied (+€3.3 ± 9.9 grouped, GradientBoosting better in only
  10 of 15 folds; +€3.8 ± 9.1 random, 13/15) and fits faster (1.4 s vs 2.5 s). The choice between
  them is not conclusive; the rule picks GradientBoosting, and the test set confirms the result
  (section 15) but does not decide it.
- Final configuration: `loss='huber'`, `n_estimators=537`, `learning_rate=0.1449`, `max_depth=2`,
  `min_samples_leaf=2`, `subsample=0.977`, `max_features=0.615`, `min_frequency=10`.

In [ ]:
candidates = ["GradientBoosting", "HistGradientBoosting", "RandomForest", "Ridge"]
rows = {}
for name in candidates:
    for protocol in ["random", "grouped"]:
        folds = final_cv[(name, protocol)]
        rows[(name, protocol)] = {
            "mae": folds["mae"].mean(), "mae_std": folds["mae"].std(), "rmse": folds["rmse"].mean(),
            "medae": folds["medae"].mean(), "fit_seconds": folds["fit_seconds"].mean(),
        }
print(pd.DataFrame(rows).T.round(2).to_string())

selected = min(candidates, key=lambda n: final_cv[(n, "grouped")]["mae"].mean())
print("\nselected (lowest grouped CV MAE):", selected)
for protocol in ["random", "grouped"]:
    ref = final_cv[(selected, protocol)]["mae"]
    for name in candidates:
        if name != selected:
            diff = final_cv[(name, protocol)]["mae"] - ref
            print(f"{protocol:8s} {name:22s} MAE - {selected}: mean {diff.mean():6.1f} | std {diff.std():5.1f} "
                  f"| {selected} better in {(diff > 0.01).sum()}/{len(diff)}")

## 15. Final Evaluation and Limitations

**Overview.** The selected model is fitted on the training set and evaluated once on the 255
held-out laptops. Residuals and errors by price range show where it is less reliable.

**Objective.** Report unbiased performance and state the model's limits with numbers.

**Key activities.**
- Test-set metrics against a median-price baseline.
- Residual diagnostics, and error by price range on the test set and on grouped out-of-fold
  predictions for the training set (which has more laptops in the expensive tail).
- Inspection of the largest errors.

**Results.**

| Metric | Final model | Median baseline |
|---|---|---|
| MAE | €152.7 | €521.4 |
| RMSE | €232.8 | €698.5 |
| R² | 0.88 | -0.04 |
| Median absolute error | €97.3 | €423.5 |
| MAPE | 13.9% | 61.2% |

- Test MAE (€152.7) is close to the random CV estimate (€157.4). This is expected: the test
  split is random and shares product families with the training set (section 06). For products
  the model has not seen, grouped CV (€179.5) and out-of-fold MAE (€176.3) are the better guide.
- The mean residual is -€0.9, so there is no overall bias, but residuals are skewed (-0.85): the
  long tail is under-prediction.

| Actual price | Test: n / MAE / bias | Out-of-fold: n / MAE / bias / MAPE |
|---|---|---|
| < €500 | 47 / €59 / +€43 | 186 / €67 / +€40 / 18.8% |
| €500–1,000 | 84 / €105 / +€59 | 339 / €108 / +€45 / 14.3% |
| €1,000–1,500 | 64 / €161 / -€8 | 250 / €172 / +€58 / 14.0% |
| €1,500–2,500 | 48 / €248 / -€55 | 202 / €272 / -€181 / 14.2% |
| ≥ €2,500 | 12 / €428 / -€341 | 43 / €767 / -€584 / 23.0% |

(bias = mean of predicted minus actual)

**Limitations.**
- **Above about €2,500 the model is less reliable.** It under-predicts systematically (bias
  -€341 on the test set, -€584 out-of-fold) and the out-of-fold relative error rises to 23%
  (14–19% elsewhere). Only 43 of 1,020 training laptops (4.2%) and 12 test laptops fall in this
  range, so the test figures there are indicative only. The Streamlit app warns when an estimate
  reaches €2,500.
- Cheaper laptops (below €1,000) are slightly over-priced (+€40 to +€59).
- Absolute error grows with price (MAE €59 below €500, €248 between €1,500 and €2,500), while the
  relative error stays around 13–16% on the test set.
- Nine of the ten largest out-of-fold errors are under-predicted high-end laptops
  (€2,858–€6,099, e.g. Razer Blade Pro, HP ZBook 17, Lenovo ThinkPad P51). The tenth is the Asus
  G701VO listed at €1,279 although the rest of its family costs €2,799–€3,975; it is over-predicted
  by €2,194 and looks like an inconsistent listing.
- The test set is small (255 laptops), and the data are historical listings of 6th–8th generation
  Intel hardware. The model is a demonstration, not a current pricing tool.

In [ ]:
X_test, y_test = test_df.drop(columns=[TARGET]), test_df[TARGET]

# Refit on the full training set; the test set plays no part in fitting or selection.
final_model = clone(tuned[selected][0].best_estimator_).fit(X_train, y_train)
test_pred = final_model.predict(X_test)

test_metrics = regression_metrics(y_test, test_pred)
baseline_metrics = regression_metrics(
    y_test, DummyRegressor(strategy="median").fit(X_train, y_train).predict(X_test))
print(pd.DataFrame({"final model": test_metrics, "median baseline": baseline_metrics}).round(2).to_string())
print(f"\nCV reference for {selected}: random MAE {final_cv[(selected, 'random')]['mae'].mean():.1f} | "
      f"grouped MAE {final_cv[(selected, 'grouped')]['mae'].mean():.1f} | test MAE {test_metrics['mae']:.1f}")

In [ ]:
oof_pred = cross_val_predict(clone(final_model), X_train, y_train,
                             cv=make_cv(replace(config, n_repeats=1), X_train, product_groups))
print("Out-of-fold MAE on train (grouped):", round(regression_metrics(y_train, oof_pred)["mae"], 1))

residuals = test_pred - y_test.to_numpy()
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
axes[0].scatter(y_test, test_pred, alpha=0.5)
limit = [0, max(y_test.max(), test_pred.max()) * 1.05]
axes[0].plot(limit, limit, color="red", linewidth=1)
axes[0].set(title="Actual vs predicted (test)", xlabel="Actual price (EUR)", ylabel="Predicted price (EUR)")
axes[1].hist(residuals, bins=30)
axes[1].axvline(0, color="red", linewidth=1)
axes[1].set(title="Residual distribution (test)", xlabel="Predicted - actual (EUR)")
axes[2].scatter(test_pred, residuals, alpha=0.5)
axes[2].axhline(0, color="red", linewidth=1)
axes[2].set(title="Residuals vs predicted (test)", xlabel="Predicted price (EUR)", ylabel="Residual (EUR)")
plt.tight_layout()
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(EXPERIMENTS_DIR / "final_evaluation.png", dpi=150)
plt.show()
print("Mean residual (bias):", round(residuals.mean(), 1), "| residual skewness:", round(pd.Series(residuals).skew(), 2))

In [ ]:
print("Test set by actual price range:\n", error_by_price_range(y_test, test_pred).round(1).to_string())
print("\nTrain out-of-fold (grouped) by actual price range:\n",
      error_by_price_range(y_train, oof_pred).round(1).to_string())

In [ ]:
columns_to_show = ["Company", "Product", "TypeName", "Ram", "Cpu", "Gpu"]
oof_table = train_df[columns_to_show].assign(
    actual=y_train, predicted=oof_pred.round(0), abs_error=np.abs(oof_pred - y_train).round(0))
print("Largest out-of-fold errors on train:\n",
      oof_table.sort_values("abs_error", ascending=False).head(10).to_string())

## 16. Model Interpretation

**Overview.** Permutation importance of the 11 raw input columns for the final model on the test
set: each column is shuffled 20 times and the increase in MAE is measured.

**Objective.** See which inputs the model relies on, and whether this agrees with the exploration.

**Key activities.** Shuffle one raw column at a time, score the whole pipeline in euros, rank the
columns by MAE increase.

**Findings.**
- Largest increases: RAM (+€79), Product (+€77), GPU (+€68), weight (+€59) and CPU (+€50). Storage
  (+€36) and type (+€35) follow; the operating system matters least (+€6).
- RAM, product and GPU agree with the exploration (section 07). RAM and Product are within noise of
  each other (standard deviation €7–8).
- Weight has almost no marginal correlation with price (-0.05), yet shuffling it costs €59, so the
  model uses it in combination with other features, such as type and screen size. This was not
  analysed further.
- Storage ranks lower than its correlation with price (0.70) suggests, partly because RAM, storage
  and resolution overlap (section 07.3).

**Caveats.** Importance is measured on raw columns, each of which feeds several derived features
(`Gpu` feeds the family and the model, `Memory` feeds four capacities), so correlated columns share
importance. It is computed for one fitted model on 255 test laptops, and it describes what the model
uses, not what causes price.

In [ ]:
perm = permutation_importance(final_model, X_test[RAW_FEATURE_COLUMNS], y_test,
                              scoring="neg_mean_absolute_error", n_repeats=20,
                              random_state=config.random_state)
importance = (pd.DataFrame({"mae_increase": perm.importances_mean, "std": perm.importances_std},
                           index=RAW_FEATURE_COLUMNS).sort_values("mae_increase", ascending=False))
print(importance.round(1).to_string())

## 17. Serialization and Experiment Tracking

**Overview.** Every evaluated model is logged to a local MLflow store, and the final model is
saved with the files the Streamlit app needs.

**Objective.** Keep a reproducible record of the experiments, and hand the app a model that
matches this notebook.

**Key activities.**
- Log 23 MLflow runs: 9 untuned models (random CV), 5 untuned models (grouped CV), the 4 tuned
  models under both protocols, and the final model with its test metrics and diagnostic files.
- Refit the selected configuration on all 1,275 laptops and save it with the input options and
  the model metadata in `models/`.
- Run the saved model through the same code path the app uses.

**Decisions.**
- Runs, parameters and metrics are stored in `experiments/mlflow.db`, and files in
  `experiments/mlartifacts` (browse with `mlflow ui --backend-store-uri sqlite:///experiments/mlflow.db`
  from the project root). The models themselves are not stored in MLflow; the deployable model is the
  joblib file below. Re-running the logging cell adds new runs.
- The deployed model is refit on the training and test laptops together, so the test metrics above
  describe the train-only model with the same configuration. The mean difference between the two
  models' predictions on the test rows is €66.1; this is a sanity check, not an evaluation.
- `models/` contains `laptop_price_model.joblib`, `input_options.json` and `model_metadata.json`. The
  metadata includes the typical out-of-fold error per predicted-price band (from €58 below €500 to
  €720 above €2,500, based on 23 laptops), which the app quotes next to each estimate.
- The saved pipeline refers to `utils.LaptopFeatureExtractor` by name, so `utils.py` must be importable
  wherever the app runs, and the model must be saved again if that class is renamed or moved.

In [ ]:
tracker = ExperimentTracker(common_params={
    "feature_set": FROZEN_SET, "target": "raw_price", "random_state": config.random_state,
    "cv_n_splits": config.n_splits, "cv_n_repeats": config.n_repeats,
})

for name, folds in portfolio.items():
    tracker.log_run(f"{name} | untuned | random CV", models[name], cv_metrics(folds),
                    tags={"stage": "portfolio", "cv_protocol": "random"})

for name, folds in grouped.items():
    tracker.log_run(f"{name} | untuned | grouped CV", models[name], cv_metrics(folds),
                    tags={"stage": "portfolio", "cv_protocol": "grouped"})

for name, (search, summary) in tuned.items():
    extra = {}
    if summary:
        extra = {"search_candidates": summary["n_candidates"],
                 "search_seconds": round(summary["search_seconds"], 1),
                 "search_best_cv_mae": round(summary["best_cv_mae"], 2)}
    for protocol in ["random", "grouped"]:
        tracker.log_run(f"{name} | tuned | {protocol} CV", search.best_estimator_,
                        cv_metrics(final_cv[(name, protocol)]),
                        tags={"stage": "tuned", "cv_protocol": protocol}, extra_params=extra)

importance.to_csv(EXPERIMENTS_DIR / "permutation_importance.csv")
final_metrics = {f"test_{k}": v for k, v in test_metrics.items()}
final_metrics.update({f"baseline_test_{k}": v for k, v in baseline_metrics.items()})
final_metrics["cv_random_mae_mean"] = final_cv[(selected, "random")]["mae"].mean()
final_metrics["cv_grouped_mae_mean"] = final_cv[(selected, "grouped")]["mae"].mean()
tracker.log_run(f"FINAL | {selected}", final_model, final_metrics, tags={"stage": "final"},
                extra_params={"selected_by": "lowest grouped CV MAE",
                              "train_rows": len(X_train), "test_rows": len(X_test)},
                artifacts=[EXPERIMENTS_DIR / "final_evaluation.png",
                           EXPERIMENTS_DIR / "permutation_importance.csv"])

In [ ]:
X_all, y_all = clean_df.drop(columns=[TARGET]), clean_df[TARGET]
deployed_model = clone(final_model).fit(X_all, y_all)

shift = np.abs(deployed_model.predict(X_test) - test_pred)
print(f"Deployed vs train-only model on the test rows: mean |difference| {shift.mean():.1f} EUR "
      f"(the deployed model has seen these rows, so this is a sanity check, not an evaluation)")

metadata = build_model_metadata(
    deployed_model, n_rows=len(clean_df), test_mae=test_metrics["mae"],
    cv_mae={"random": final_cv[(selected, "random")]["mae"].mean(),
            "grouped": final_cv[(selected, "grouped")]["mae"].mean()},
    oof_true=y_train, oof_pred=oof_pred,
)
save_artifacts(deployed_model, build_input_options(clean_df), metadata)
print(pd.DataFrame(metadata["error_by_predicted_price"]).to_string())

In [ ]:
loaded = joblib.load(MODELS_DIR / MODEL_FILENAME)
options = json.loads((MODELS_DIR / OPTIONS_FILENAME).read_text(encoding="utf-8"))
row = build_laptop_row(company="Dell", type_name="Notebook", product_line="inspiron", opsys="Windows 10",
                       inches=15.6, resolution="1920x1080", touchscreen=False, ips=False, weight_kg=2.1,
                       cpu="Intel Core i5 7200U 2.5GHz", gpu="Intel HD Graphics 620",
                       ram_gb=8, ssd_gb=256, hdd_gb=0, flash_gb=0)
print(LaptopFeatureExtractor().transform(row).T.to_string(header=False))
print("Predicted price:", round(float(loaded.predict(row)[0])))
print({key: len(values) for key, values in options.items() if isinstance(values, list)})

similar = clean_df[clean_df["Product"].str.lower().str.startswith("inspiron") & (clean_df["Ram"] == "8GB")]
print("Dell-style Inspiron rows with 8GB in the data:\n", similar[TARGET].describe().round(0).to_string())

## 18. Final Summary

**Overview.** The project predicts laptop prices from their specifications and delivers a trained
model, a Streamlit app and tracked experiments.

**Results.**
- **Data:** 1,303 listings reduced to 1,275 after removing 28 redundant rows; split into 1,020
  training and 255 test laptops, stratified by price.
- **Features:** text columns parsed by a stateless extractor; frozen feature set `gpu_product`
  (base specifications, CPU and GPU family, GPU model, product line); raw price as target.
- **Model:** GradientBoosting tuned on grouped cross-validation. Cross-validated MAE is €157.4 with
  random folds and €179.5 when whole products are held out.
- **Test performance:** MAE €152.7, RMSE €232.8, R² 0.88; the MAE is about 71% below that of a median
  baseline.
- **Deployment:** the final configuration is refit on all 1,275 laptops and served by `app.py`;
  experiments are tracked with MLflow (section 17).

**Limitations.**
- Estimates above about €2,500 are less reliable: the model under-predicts there and few laptops
  were available for training.
- Cheaper laptops are slightly over-priced.
- For unseen products, expect an error closer to the grouped estimate (about €180) than to the test
  MAE.
- The data are historical listings; the model is not a current pricing tool.

**Next steps.** More laptops above €2,500 would address the main weakness; current listings would
be needed for current prices.